In [ ]:
import pandas as pd
import numpy as np
from sklearn import naive_bayes
import os
import sys
import re
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from utils import process_tweet

In [ ]:
#Reading the training data file
data = pd.read_csv("../data/processed/sentiment_label_data.csv")
data.sample(5)

In [ ]:
#cleaning the tweets 
data["tweets"] = data["text"].map(lambda x : process_tweet(x))

In [ ]:
#Added column clean_tweet to store te processed tweets 
data["tweets"].sample(10)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data['tweets'], data['sentiment'], test_size = 0.333, random_state=0)

In [ ]:
X_train[0]

In [ ]:
y_train[0]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vect = CountVectorizer().fit(X_train)


In [ ]:
vect.get_feature_names()[::50]

In [ ]:
X_train_vectorized = vect.transform(X_train)

In [ ]:
#Training the model
clfrNB = naive_bayes.MultinomialNB()
clfrNB.fit(X_train_vectorized, y_train)

In [ ]:
#Vectorizing and predicting the labels 
X_test_vectorized = vect.transform(X_test)
predicted_labels = clfrNB.predict(X_test_vectorized)

In [ ]:
# Metrics to evaluate the model
from sklearn.metrics import roc_auc_score
print("AUC:", roc_auc_score (y_test, predicted_labels))

In [ ]:
# Confusion Matrix to evaluate for binary classification
from sklearn.metrics import confusion_matrix
confusion_matrix = confusion_matrix(y_test, predicted_labels)
confusion_matrix = confusion_matrix.astype(int)
print(confusion_matrix)

In [ ]:
layout = {
    "title":"Confusion Matrix",
    "xaxis": {"title":"Predicted value"},
    "yaxis":{"title":"True value"}
} 

fig = go.Figure(data=go.Heatmap(z=confusion_matrix,
                                hoverongaps=False),
                                layout=layout)
fig.show()

In [ ]:
#Calculating precision, recall, f1_score 
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
print('Precision    : %.3f'%precision_score(y_test, predicted_labels))
print('Recall       : %.3f'%recall_score(y_test, predicted_labels))
print('F1 Score     : %.3f'%f1_score(y_test, predicted_labels))

In [ ]:
#Predicting the Tweet Sentiment using trained model
data = pd.read_csv("../data/processed/processed_tweets.csv")


In [ ]:
tweets = tesla_data["clean_tweet"]
tweets_vectorized = vect.transform(tsla_tweets)

In [ ]:
#predicting the sentiment of tweets
data["predictions"] = clfrNB.predict(tweets_vectorized)


In [ ]:
#Display postive and negative sentiment count
df = tesla_data["predictions"].value_counts()


In [ ]:
tweet_predictions_senti = tesla_data
tweet_predictions_senti["predictions"] = tweet_predictions_senti["predictions"].replace({1:'Positive'})
tweet_predictions_senti["predictions"] = tweet_predictions_senti["predictions"].replace({0:'Negative'})
fig = px.histogram(tweet_predictions_senti, x="predictions")
fig.update_traces(marker_color="indianred", marker_line_color='black')
fig.update_layout(title_text="Tweets Sentiment")
fig.show()


In [ ]:
# Storing the prediction to csv file
tesla_data.to_csv('../data/output/naive_bayes_sentiment_predictions.csv')